In [1]:
from matplotlib import pyplot as plt
import os
import sys
import numpy as np
import pandas as pd
import cv2

model_data = {
    'tasks': ['Flattening'],
    'method': 'diffusion',
    'fabrics': ['dress', 'longsleeve', 'pants', 'skirt'],
    'heatmap': False,
    'input_obs_rgb': False,
    'input_obs_depth': False,
    'denoise_obs': False,
    'grasp_img': False,
    'exp_dirs': [
        #'flattening-agent-cloth-mask-centre-multiple-prioritise-middle+RGB2M-on-mix-800k-all-garment-bonus-penalty-v2-50000'
        #'flattening-agent-cloth-mask-centre-multiple-prioritise-middle+RGB2M-on-mix-200k-sf-50000',
        '../test_results/human_data/flattening'
    ]
}
model = model_data
exp_dirs = model['exp_dirs']
tasks = model['tasks']
# method = model['method']
  
fabrics = model['fabrics']
heatmap = model['heatmap']
input_obs_rgb = model['input_obs_rgb']
input_obs_depth = model.get('input_obs_depth', False)
denoise_obs = model.get('denoise_obs', False)
grasp_img = model.get('grasp_img', False)

In [2]:
## get all direcotry not recursive in the current directory
def get_all_dir(path):
    print(path)
    ret = []
    for d in os.listdir(path):
        
        dir_path = os.path.join(path, d)
        if os.path.isdir(dir_path):
            ret.append(dir_path)

    return ret

# for exp_dir in exp_dirs:
#     experiments = get_all_dir(exp_dir)
#     experiments = sorted(experiments)
import json

def coverage_alignment_reward(NC1, NC2, IoU1, IoU2):
    dNC = NC2 - NC1
    du = IoU2 - IoU1
    alpha = 2
    beta = 1
    rca = max(np.tanh(alpha*dNC + du*beta), 0)
    b = 0.7

    if NC1 > 0.9 and NC2 < 0.9:
        return 0
    
    if NC2 >= 0.95:
        return b
    
    return rca


print_reward = True
OBJECT = 'garment'

In [3]:
## show action_images as a trajecotry in an experiment

def show_trajectory(experiment, plt_steps=None, task='Flattening'):
    print('experiment:', experiment)
    ## get all steps in the experiment, they are folders
    steps = [d for d in os.listdir('./' + experiment) if os.path.isdir('./' + experiment + '/' + d)]
    #print('steps:', steps)
    steps = [int(s.split('_')[-1]) for s in steps if 'step' in s]
    plt_steps = steps if plt_steps is None else plt_steps
    plt_steps = sorted(plt_steps)
    horizon = len(plt_steps)
    #print('horizon:', horizon)
   


    action_images = []
    rgb_images = []
    pick_heat_images = []
    place_heat_images = []
    mask_images = []
    depth_images = []
    input_obs_images = []
    input_obs_d_images = []
    normalised_coverages = []
    grasp_imgs = []
    denoise_obs_imgs = []
    IoUs = []
    print('plt_steps:', plt_steps)

    for i in plt_steps:
        print('step', i)
        rgb_images.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/workspace_rgb.png'))
        mask_images.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/mask.png'))
        depth_images.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/depth.png'))

        # it is in evaluation.json file
        
        with open(experiment + '/' + f'step_{str(i)}' + '/evaluation.json') as f:
            eval = json.load(f)
            if task == 'Flattening':
                normalised_coverages.append(eval['normalised_coverage'])
                IoUs.append(eval['max_IoU'])

        with open(experiment  + '/info.json') as f:
            info = json.load(f)
            posterior_rewards = info['posterior_rewards']
            prior_rewards = info['prior_rewards']
        
        
        if i != plt_steps[-1]:
            if input_obs_rgb:
                input_obs_images.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/input_obs_rgb.png'))
            
            if input_obs_depth:
                input_obs_d_images.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/input_obs_depth.png'))

            if denoise_obs:
                denoise_obs_imgs.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/noise_actions_obs_depth.png'))
            
            if grasp_img:
                grasp_imgs.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/grasp_image.png'))

            #action_images.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/action_obs_rgb.png'))
            action_images.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/action_image.png'))
            if heatmap:
                pick_heat_images.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/pick-heat.png'))
                place_heat_images.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/place-heat.png'))
    
    row = 6 if heatmap else 4
    if input_obs_rgb:
        row += 1
    if input_obs_depth:
        row += 1
    if denoise_obs:
        row += 1
    if grasp_img:
        row += 1
    depth_list = []
    for i, depth in enumerate(input_obs_d_images):
        d = cv2.applyColorMap(np.uint8(255 * depth), cv2.COLORMAP_JET)
        d = cv2.cvtColor(d, cv2.COLOR_BGR2RGB)
        depth_list.append(d)
    input_obs_d_images = depth_list

    plt_horizon = max(10, horizon)
    plt_horizon = horizon #min(plt_horizon, 20)
    fig, axs = plt.subplots(row, horizon, figsize=(plt_horizon*4, (4)*row))
    for i in range(horizon):
        axs[0, i].imshow(rgb_images[i])
        axs[1, i].imshow(mask_images[i])
       

        axs[2, i].imshow(1.0-depth_images[i], cmap='cividis')
        
        if i < horizon-1:
            axs[-1, i].imshow(action_images[i])
            b = 0

            
            
            if grasp_img:
               
                axs[-2-b, i].imshow(grasp_imgs[i])
                b += 1
            
            if denoise_obs:
                
                axs[-2-b, i].imshow(denoise_obs_imgs[i])
                b += 1

            if input_obs_rgb:
                axs[-2-b, i].imshow(input_obs_images[i])
                b += 1
            if input_obs_depth:
                #axs[-2-b, i].imshow(input_obs_d_images[i])
                ## colormap fancy
                axs[-2-b, i].imshow(input_obs_d_images[i], cmap='cividis')
                b += 1
            
            

            if heatmap:
                axs[-2-b, i].imshow(pick_heat_images[i])
                axs[-3-b, i].imshow(place_heat_images[i])
        else:
            axs[-1, i].imshow(rgb_images[-1])
        
        
        axs[0, i].axis('off')
        axs[1, i].axis('off')
        axs[2, i].axis('off')
        axs[-1, i].axis('off')
        b = 0
        print('denoise_obs:', denoise_obs)
        if grasp_img:
            axs[-2-b, i].axis('off')
            b +=1
        if denoise_obs:
            axs[-2-b, i].axis('off')
            b +=1
        

        if input_obs_rgb:
            axs[-2-b, i].axis('off')
            b +=1   
        if input_obs_depth:
            axs[-2-b, i].axis('off')
            b +=1
       

        if heatmap:
            axs[-2-b, i].axis('off')
            axs[-3-b, i].axis('off')
    
    # Add text on the first image of each row
    label_fontsize = 18
    label_color = 'white'  # Change to white for better visibility on dark images
    label_bgcolor = 'black'  # Background color for contrast
    label_alpha = 0.7
    label_pad = 5

    def add_text(ax, text, pos='bottom'):
        if pos == 'bottom':
            ax.text(0.5, 0.05, text, fontsize=label_fontsize, color=label_color,
                    transform=ax.transAxes, va='bottom', ha='center',
                    bbox=dict(facecolor=label_bgcolor, alpha=label_alpha, edgecolor='none', pad=label_pad))
        else:
            ax.text(0.5, 0.95, text, fontsize=label_fontsize, color=label_color,
                    transform=ax.transAxes, va='top', ha='center',
                    bbox=dict(facecolor=label_bgcolor, alpha=label_alpha, edgecolor='none', pad=label_pad))

    add_text(axs[0, 0], 'RGB')
    add_text(axs[1, 0], 'Mask')
    add_text(axs[2, 0], 'Depth')
    #add_text(axs[-1, 0], 'Action on Input RGB')
    for i in range(horizon):
        if task == 'Flattening':
            string2print = f'NC: {int(normalised_coverages[i]*100)}'
            if OBJECT == 'garment':
                string2print += f', IoU: {int(IoUs[i]*100)}'
            
            if print_reward == True and i > 0 and i < horizon:
                reward = coverage_alignment_reward(
                    normalised_coverages[i-1], 
                    normalised_coverages[i],
                    IoUs[i-1],
                    IoUs[i])
                string2print += f'\n GT-R: {int(reward*100)}, PST-R: {int(posterior_rewards[i]*100)}, PR-R: {int(prior_rewards[i-1]*100)},'

            add_text(axs[-1, i], string2print, pos='bottom')
        else:
            add_text(axs[-1, i], f'IoU: {IoUs[i]:.2f}', pos='bottom')
    

    b = 0

    
    if grasp_img:
        add_text(axs[-2-b, 0], 'Grasping')
        b +=1

    if denoise_obs:
        add_text(axs[-2-b, 0], 'Denoising')
        b +=1

    if input_obs_rgb:
        add_text(axs[-2-b, 0], 'Input RGB')
        
        b +=1
    if input_obs_depth:
        add_text(axs[-2-b, 0], 'Input Depth')
        b +=1
   
    if heatmap:
        add_text(axs[-2-b, 0], 'Pick Heat')
        add_text(axs[-3-b, 0], 'Place Heat')
        
    
    ## remove the inter-space between subplots
    #plt.subplots_adjust(wspace=0, hspace=0)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0, hspace=0)
    fig.tight_layout(pad=0)

    # make it tight, column inter-space is 0
    #plt.tight_layout()
    plt.savefig(experiment + '/trajectory.png')
    plt.show()



In [4]:
## Get step-wise evaluation results
import json
def get_stepwise_eval(experiment):
    steps = [d for d in os.listdir('./' + experiment) if os.path.isdir('./' + experiment + '/' + d)]
    ## filter the steps that have `goal` in the name
    steps = [s for s in steps if 'step' in s]
    #print('steps:', steps)
    horizon = len(steps)
    #print('horizon:', horizon)
    evaluations = {
        'trial': [],
        'fabric': [],
        'state': [],
    }
    
    for i in range(horizon):
        eval_file = experiment + '/' + f'step_{str(i)}' + '/evaluation.json'
        if not os.path.exists(eval_file):
            print('evaluation file not found:', eval_file)
            continue
        evaluations['fabric'].append(experiment.split('/')[-1].split('-')[0])
        evaluations['trial'].append(experiment.split('/')[-1].split('-')[-1])
        evaluations['state'].append(i)
        with open(eval_file, 'r') as f:
            eval_data = json.load(f)
            for k, v in eval_data.items():
                if k not in evaluations:
                    evaluations[k] = []
                evaluations[k].append(v)
    df = pd.DataFrame(evaluations)
    df['steps'] = horizon - 1
    return df

totoal_stats_IoU = []
total_human_success = []
total_stats_nc = []
total_stats_ni = []
total_stats_s2f = []
for task, exps in zip(tasks, exp_dirs):
    print('############################################\n\n')
    print(task)
    experiments = get_all_dir(exps)
    evals = []
    for exp in experiments:
        evals.append(get_stepwise_eval(exp))

    eval_df = pd.concat(evals)

    

    #display(eval_df)

    # Sort by trial, fabric, and state (assuming state is sortable and increasing)
    df_sorted = eval_df.sort_values(['trial', 'fabric', 'state'])

    # Define a function to apply to each group
    def get_first_or_last(group):
        # Find rows that meet both conditions
        filtered = group[(group['normalised_coverage'] > 0.9) & (group['max_IoU'] > 0.8)]
        if not filtered.empty:
            # If any row meets the criteria, return the first one
            return filtered.iloc[0]
        else:
            # Otherwise, return the last row in the group
            return group.iloc[-1]

    # Apply the function to each (trial, fabric) group
    result = df_sorted.groupby(['trial', 'fabric'], group_keys=False).apply(get_first_or_last).reset_index(drop=True)
    result.loc[
    (result['normalised_coverage'] > 0.9) & (result['max_IoU'] > 0.8),
    'auto success'
    ] = True
    display(result)
    eval_df = result
    


    if 'max_IoU' in eval_df.columns:
        # IoU x 100
        eval_df['max_IoU'] = eval_df['max_IoU'] * 100
        #display(eval_df)

        # for each fabric trial, get the largest IoU

        IoU_stats_df = eval_df.groupby(['fabric', 'trial']).agg({'max_IoU': 'max'}).reset_index()
        totoal_stats_IoU.append(IoU_stats_df)
        # for each fabric print the mena and std of IoU from IoU_stats_df
        IoU_stats_df = IoU_stats_df.groupby('fabric').agg({'max_IoU': ['mean', 'std']}).reset_index()

       

    if 'normalised_coverage' in eval_df.columns:
        eval_df['normalised_coverage'] = eval_df['normalised_coverage'] * 100
        # get the row for the last state for eah fabric trial, and get the mean and std of normalised_coverage for each fabric
        nc_stats_df = eval_df[eval_df['normalised_coverage'] < 101.0]
        nc_stats_df = nc_stats_df.groupby(['fabric', 'trial']).agg({'normalised_coverage': 'max'}).reset_index()
        total_stats_nc.append(nc_stats_df)
        nc_stats_df = nc_stats_df.groupby('fabric').agg({'normalised_coverage': ['mean', 'std']}).reset_index()

 

    if 'normalised_improvement' in eval_df.columns:
        eval_df['normalised_improvement'] = eval_df['normalised_improvement'] * 100
        ni_stats_df = eval_df[eval_df['normalised_improvement'] < 101.0]
        ni_stats_df = ni_stats_df.groupby(['fabric', 'trial']).agg({'normalised_improvement': 'max'}).reset_index()
        total_stats_ni.append(ni_stats_df)
        ni_stats_df = ni_stats_df.groupby('fabric').agg({'normalised_improvement': ['mean', 'std']}).reset_index()


    if 'steps' in eval_df.columns:
        #eval_df['normalised_improvement'] = eval_df['normalised_improvement'] * 100
        
        s2f_stats_df = eval_df.groupby(['fabric', 'trial']).agg({'steps': 'min'}).reset_index()
        total_stats_s2f.append(s2f_stats_df)
        s2f_stats_df = s2f_stats_df.groupby('fabric').agg({'steps': ['mean', 'std']}).reset_index()


    ## display steps statistics mean and std
    print('Steps statistics')
    display(eval_df.groupby('fabric').agg({'steps': ['mean', 'std']}))
    
    print('IoU statistics')
    display(eval_df.groupby('fabric').agg({'max_IoU': ['mean', 'std']}))

    print('NC statistics')
    display(eval_df.groupby('fabric').agg({'normalised_coverage': ['mean', 'std']}))

    print('NI statistics')
    display(eval_df.groupby('fabric').agg({'normalised_improvement': ['mean', 'std']}))

    ## total statistics for steps
    print('Total steps statistics')
    display(eval_df.agg({'steps': ['mean', 'std']}))
    # get the largest 'state' for each fabric trial
    auto_success_df = eval_df.groupby(['fabric', 'trial']).agg({'auto success': 'max'}).reset_index()
    # count the number of True for each fabric
    auto_success_df = auto_success_df.groupby('fabric').agg({'auto success': 'sum'}).reset_index()

    #display(human_success_df)
    # get the number of trials for each fabric
    num_trials_df = eval_df.groupby('fabric').agg({'trial': 'nunique'}).reset_index()

    # combine the two dataframes
    auto_success_df = auto_success_df.merge(num_trials_df, on='fabric')
    auto_success_df.columns = ['fabric', 'num_human_success', 'num_trials']
    total_human_success.append(auto_success_df)
    display(auto_success_df)


## get total stats


if len(total_stats_nc) > 0:
    total_stats_nc = pd.concat(total_stats_nc)
    print('\n\nTotal normalised coverage stats')
    display(total_stats_nc.agg({'normalised_coverage': ['mean', 'std']}))

if len(total_stats_ni) > 0:
    total_stats_ni = pd.concat(total_stats_ni)
    print('\n\nTotal normalised improvement stats')
    display(total_stats_ni.agg({'normalised_improvement': ['mean', 'std']}))

if len(totoal_stats_IoU) > 0:
    totoal_stats_IoU = pd.concat(totoal_stats_IoU)
    print('\n\nTotal IoU stats')
    display(totoal_stats_IoU.agg({'max_IoU': ['mean', 'std']}))

if len(total_stats_s2f) > 0:
    totoal_stats_IoU = pd.concat(total_stats_s2f)
    print('\n\nTotal S2F')
    display(totoal_stats_IoU.agg({'steps': ['mean', 'std']}))


total_human_success = pd.concat(total_human_success)
print('\n\nTotal human success')
display(total_human_success.agg({'num_human_success': 'sum', 'num_trials': 'sum'}))


############################################


Flattening
../test_results/human_data/flattening


,trial,fabric,state,max_coverage,init_coverage,coverage,normalised_coverage,normalised_improvement,max_IoU,auto success,human success,steps
0,0,dress,2,202531,167671,192091,0.948452,0.700516,0.836103,True,False,6
1,0,longsleeve,9,152713,60703,139548,0.913793,0.856918,0.863090,True,False,11
2,0,pants,8,102406,40998,103486,1.000000,1.000000,0.901766,True,False,9
3,0,skirt,6,150001,113469,155016,1.000000,1.000000,0.827185,True,False,7
4,1,dress,16,207589,117166,188165,0.906430,0.785187,0.887783,True,False,17
5,1,longsleeve,5,154422,70053,143533,0.929485,0.870936,0.844883,True,False,9
6,1,pants,20,104050,45084,81568,0.783931,0.618729,0.673673,False,True,20
7,1,skirt,3,158404,66940,147819,0.933177,0.884271,0.882764,True,True,3
8,2,dress,14,207955,78316,184025,0.884927,0.815410,0.767877,False,True,14
9,2,longsleeve,5,152736,75618,146189,0.957135,0.915104,0.854631,True,False,8


Steps statistics


steps          
            mean       std
fabric                    
dress       15.9  5.839521
longsleeve  10.6  3.893014
pants       10.7  4.989990
skirt        5.9  2.424413

IoU statistics


max_IoU          
                 mean       std
fabric                         
dress       75.035115  9.680809
longsleeve  84.010008  4.058120
pants       84.655718  7.125987
skirt       86.789612  3.631851

... (13 outputs left out)



Total S2F


,steps
mean,10.77500
std,5.59527




Total human success


num_human_success    31
num_trials           40
dtype: int64

In [5]:

for task, exps in zip(tasks, exp_dirs):
    print('############################################\n\n')
    print(task)
    experiments = get_all_dir(exps)

    for exp in experiments:
        show_trajectory(exp, task=task)#, plt_steps=[0, 1, 2, 3, 5, 6, 7])


############################################


Flattening
../test_results/human_data/flattening
experiment: ../test_results/human_data/flattening/skirt-4
plt_steps: [0, 1, 2, 3, 4, 5]
step 0


FileNotFoundError: [Errno 2] No such file or directory: '../test_results/human_data/flattening/skirt-4/step_0/workspace_rgb.png'

In [ ]:
from fpdf import FPDF
import pandas as pd
import csv
import io

pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.set_font("Arial", size=12)

for task, exps in zip(tasks, exp_dirs):
    print('############################################\n\n')
    print(task)
    experiments = get_all_dir(exps)

    for i, exp in enumerate(experiments):
        pdf.add_page()
        # add text of exp
        text = exp.split('/')[-1]
        pdf.cell(200, 10, txt=text, ln=True, align='C')
        # let the image fit the page, considering both width and height
        pdf.image(exp + '/trajectory.png', x=10, y=20, w=190)

        ## add evaluation results
        pdf.add_page()
        pdf.cell(200, 10, txt='Evaluation', ln=True, align='C')
        
        # Convert DataFrame to CSV string
        csv_buffer = io.StringIO()
        # get rid of fabric and trial columns
        eval_df = get_stepwise_eval(exp)
        eval_df = eval_df.drop(columns=['fabric', 'trial'])
        eval_df.to_csv(csv_buffer, index=False)
        csv_string = csv_buffer.getvalue()
        
        # Parse CSV string
        csv_reader = csv.reader(csv_string.splitlines())
        
        # Set a smaller font size for the table
        pdf.set_font("Arial", size=10)
        
        # Calculate column widths and overall table width
        data = list(csv_reader)
        col_widths = [max(len(str(row[i])) for row in data) * 2.5 for i in range(len(data[0]))]
        ## add the first column width
        #col_widths.insert(0, 5)
        table_width = sum(col_widths)
        
        # Scale column widths if table is too wide
        page_width = pdf.w - 20  # Assuming 10mm margin on each side
        if table_width > page_width:
            scale = page_width / table_width
            col_widths = [w * scale for w in col_widths]
        
        # Print CSV data
        for j, row in enumerate(data):
            pdf.ln(10)  # Add some vertical space between rows
            #pdf.cell(col_widths[0], 10, str(j-1), border=1, align='C')
            for i, col in enumerate(row):
                # if it can be converted to float, then print it with 2 decimal points
                if col.replace('.', '', 1).isdigit():
                    val = float(col)
                    pdf.cell(col_widths[i], 10, f"{val:.2f}", border=1, align='C')
                else:
                    pdf.cell(col_widths[i], 10, str(col), border=1, align='C')
        
        # Reset font size
        pdf.set_font("Arial", size=12)

    pdf.output(f"files/{exps}.pdf")


ModuleNotFoundError: No module named 'fpdf'